# 📤 TikTok Ads - Data Upload Tool

**Simple tool for uploading TikTok Ads data to database**

## How to use:
1. Set `UPLOAD_FILE` path in Cell 2
2. Run all cells
3. Check upload summary

---

In [ ]:
# ============================================================
# CELL 2: CONFIGURATION
# ============================================================
import os

# Tenant configuration
TENANT_ID = "yumna_bertigamart"

# ⚠️ SET YOUR FILE PATH HERE:
UPLOAD_FILE = None  # Example: "C:/Users/yumna/Downloads/tiktok_ads_jan2026.xlsx"

# Upload mode: "skip" = skip duplicates, "update" = update existing
UPLOAD_MODE = "skip"

# Auto-detect database path
DB_DIR = None
for path in [
    "../backend/config/databases",
    "backend/config/databases",
    "C:/Users/yumna/Desktop/Project/omni/backend/config/databases"
]:
    if os.path.exists(os.path.abspath(path)):
        DB_DIR = os.path.abspath(path)
        break

DB_PATH = f"{DB_DIR}/{TENANT_ID}.db" if DB_DIR else None

print("📋 Upload Configuration:")
print(f"   Tenant: {TENANT_ID}")
print(f"   Database: {DB_PATH}")
print(f"   File: {UPLOAD_FILE or '❌ NOT SET - Edit this cell!'}")
print(f"   Mode: {UPLOAD_MODE}")

In [ ]:
# ============================================================
# CELL 3: IMPORTS
# ============================================================
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import hashlib

print("✅ Libraries loaded")

In [ ]:
# ============================================================
# CELL 4: UPLOAD FUNCTION
# ============================================================

def upload_tiktok_ads(file_path: str, tenant_id: str, db_path: str, mode: str = "skip"):
    """
    Upload TikTok Ads data from Excel/CSV to database.
    
    Args:
        file_path: Path to Excel/CSV file
        tenant_id: Tenant ID
        db_path: Path to SQLite database
        mode: "skip" to skip duplicates, "update" to update existing
    
    Returns:
        dict with upload statistics
    """
    if not file_path:
        print("❌ ERROR: UPLOAD_FILE not set!")
        print("   Please edit Cell 2 and set UPLOAD_FILE path")
        return None
    
    if not os.path.exists(file_path):
        print(f"❌ ERROR: File not found: {file_path}")
        return None
    
    print(f"📤 Starting upload...")
    print(f"   File: {file_path}")
    print(f"   Mode: {mode}")
    
    # Read file
    file_ext = Path(file_path).suffix.lower()
    if file_ext in ['.xlsx', '.xls']:
        df = pd.read_excel(file_path)
    elif file_ext == '.csv':
        df = pd.read_csv(file_path)
    else:
        print(f"❌ ERROR: Unsupported format: {file_ext}")
        return None
    
    print(f"   Rows read: {len(df):,}")
    
    # Column mapping (TikTok Indonesian export → database)
    column_mapping = {
        'ID Campaign': 'campaignId',
        'Nama Campaign': 'campaignName',
        'ID Produk': 'productId',
        'Jenis Iklan': 'creativeType',
        'Judul Video': 'videoTitle',
        'ID video': 'videoId',
        'Akun TikTok': 'tiktokAccount',
        'Status': 'status',
        'Jenis Otorisasi': 'authorizationType',
        'Biaya (IDR)': 'cost',
        'Pesanan (SKU)': 'ordersSku',
        'Biaya Per Pesanan (CPO)': 'costPerOrder',
        'Pendapatan Kotor (IDR)': 'grossRevenue',
        'ROI': 'roi',
        'Impressions': 'impressions',
        'Clicks': 'clicks',
        'CTR': 'ctr',
        'Conversion Rate': 'conversionRate',
        'Mulai (Metrics)': 'periodStart',
        'Sampai (Metrics)': 'periodEnd',
        'Tingkat Tontonan Video: 2 detik': 'watchRate2s',
        'Tingkat Tontonan Video: 6 detik': 'watchRate6s',
        'Tingkat Tontonan Video: 25%': 'watchRate25pct',
        'Tingkat Tontonan Video: 50%': 'watchRate50pct',
        'Tingkat Tontonan Video: 75%': 'watchRate75pct',
        'Tingkat Tontonan Video: 100%': 'watchRate100pct',
    }
    
    df = df.rename(columns=column_mapping)
    
    # Add metadata
    batch_id = f"notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    df['tenantId'] = tenant_id
    df['uploadBatchId'] = batch_id
    df['currency'] = 'IDR'
    df['createdAt'] = datetime.now().isoformat()
    df['updatedAt'] = datetime.now().isoformat()
    
    # Parse dates
    for col in ['periodStart', 'periodEnd']:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')
    
    # Create unique hash for deduplication
    def create_hash(row):
        key = f"{row.get('campaignId', '')}_{row.get('productId', '')}_{row.get('videoId', '')}_{row.get('periodStart', '')}_{row.get('periodEnd', '')}"
        return hashlib.md5(key.encode()).hexdigest()[:16]
    
    df['uniqueHash'] = df.apply(create_hash, axis=1)
    
    # Database columns
    db_columns = [
        'tenantId', 'uploadBatchId', 'periodStart', 'periodEnd',
        'campaignId', 'campaignName', 'productId', 'creativeType',
        'videoTitle', 'videoId', 'tiktokAccount', 'status', 'authorizationType',
        'cost', 'ordersSku', 'costPerOrder', 'grossRevenue', 'roi',
        'impressions', 'clicks', 'ctr', 'conversionRate',
        'watchRate2s', 'watchRate6s', 'watchRate25pct',
        'watchRate50pct', 'watchRate75pct', 'watchRate100pct',
        'currency', 'uniqueHash', 'createdAt', 'updatedAt'
    ]
    
    for col in db_columns:
        if col not in df.columns:
            df[col] = None
    
    df = df[db_columns]
    
    # Connect to database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Get existing hashes
    cursor.execute("SELECT uniqueHash FROM TiktokAdsCreativeData WHERE tenantId = ?", (tenant_id,))
    existing_hashes = set(row[0] for row in cursor.fetchall() if row[0])
    
    # Filter
    new_records = df[~df['uniqueHash'].isin(existing_hashes)]
    duplicate_records = df[df['uniqueHash'].isin(existing_hashes)]
    
    inserted = 0
    updated = 0
    skipped = len(duplicate_records)
    
    # Insert new records
    if len(new_records) > 0:
        placeholders = ','.join(['?' for _ in db_columns])
        insert_sql = f"INSERT INTO TiktokAdsCreativeData ({','.join(db_columns)}) VALUES ({placeholders})"
        
        for _, row in new_records.iterrows():
            cursor.execute(insert_sql, tuple(row[col] for col in db_columns))
            inserted += 1
    
    # Update existing if mode is 'update'
    if mode == 'update' and len(duplicate_records) > 0:
        for _, row in duplicate_records.iterrows():
            update_cols = [c for c in db_columns if c not in ['tenantId', 'uniqueHash', 'createdAt']]
            set_clause = ', '.join([f"{c} = ?" for c in update_cols])
            update_sql = f"UPDATE TiktokAdsCreativeData SET {set_clause} WHERE uniqueHash = ? AND tenantId = ?"
            values = [row[c] for c in update_cols] + [row['uniqueHash'], tenant_id]
            cursor.execute(update_sql, values)
            updated += 1
            skipped -= 1
    
    # Create batch record
    cursor.execute("""
        INSERT INTO TiktokAdsUploadBatch 
        (id, tenantId, fileName, totalRows, insertedRows, updatedRows, skippedRows, status, createdAt)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        batch_id, tenant_id, Path(file_path).name,
        len(df), inserted, updated, skipped, 'completed', datetime.now().isoformat()
    ))
    
    conn.commit()
    conn.close()
    
    print("\n" + "="*50)
    print("✅ UPLOAD COMPLETE!")
    print("="*50)
    print(f"   📁 File: {Path(file_path).name}")
    print(f"   📊 Total Rows: {len(df):,}")
    print(f"   ✨ Inserted: {inserted:,}")
    print(f"   🔄 Updated: {updated:,}")
    print(f"   ⏭️  Skipped: {skipped:,}")
    print(f"   🏷️  Batch ID: {batch_id}")
    
    return {
        "status": "success",
        "batch_id": batch_id,
        "total": len(df),
        "inserted": inserted,
        "updated": updated,
        "skipped": skipped
    }

In [ ]:
# ============================================================
# CELL 5: EXECUTE UPLOAD
# ============================================================

if UPLOAD_FILE:
    result = upload_tiktok_ads(UPLOAD_FILE, TENANT_ID, DB_PATH, UPLOAD_MODE)
else:
    print("⚠️ No file to upload.")
    print("")
    print("To upload data:")
    print("1. Edit Cell 2")
    print("2. Set UPLOAD_FILE = 'C:/path/to/your/file.xlsx'")
    print("3. Run all cells again")

In [ ]:
# ============================================================
# CELL 6: CHECK CURRENT DATA
# ============================================================

conn = sqlite3.connect(DB_PATH)

# Count records
total = pd.read_sql_query(
    "SELECT COUNT(*) as cnt FROM TiktokAdsCreativeData WHERE tenantId = ?",
    conn, params=(TENANT_ID,)
)['cnt'].iloc[0]

# Recent batches
batches = pd.read_sql_query("""
    SELECT id, fileName, totalRows, insertedRows, skippedRows, createdAt
    FROM TiktokAdsUploadBatch 
    WHERE tenantId = ?
    ORDER BY createdAt DESC
    LIMIT 5
""", conn, params=(TENANT_ID,))

conn.close()

print(f"📊 Current Data Status:")
print(f"   Total Records: {total:,}")
print(f"")
print("📦 Recent Upload Batches:")
display(batches)